In [ ]:
from typing import Tuple, List, Optional
import numpy as np
np.__version__


In [ ]:
def creating_data(
    w: list[float],
    x_min: float = -40.0,
    x_max: float = 40.0,
    data_size: int = 100,
    noise_std: float = 0.0,
    seed: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """Вернёт данные разметки с нормальным распределением ошибки измерений.

    Parameters:
        w: вектор (w_0, w_1) с реальными параметрами модели.
        x_min: (по умол. -40.0), минимальное значение входных данных.
        x_max: (по умол. 40.0), максимальное значение входных данных.
        data_size: (по умол. 100), кол-во наблюдений в наборе данных.
        noise_std: (по умол. 0.0), стандартное отклонение ошибки.
        seed: (по умол. None), значение для инициализации генератора 
    случайных чисел.

    Returns:
        Tuple[np.ndarray, np.ndarray]
    """
    rng = np.random.default_rng(seed)

    x = np.linspace(x_min, x_max, data_size, dtype=np.float32)  # (data_size,)
    xs = np.stack((np.ones_like(x), x), axis=1)   # (data_size, 2)
    ys = xs @ np.array(w, dtype=np.float32)    # (data_size,)

    # Добавляем ошибку в данные.
    if noise_std > 0:
        ys += rng.normal(scale=noise_std, size=ys.shape).astype(np.float32)
        
    return xs, ys


def linear_model(x: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Линейная модель f(x) = x @ w.T."""
    return x @ w.T


def train(
    model,
    xs: np.ndarray,
    ys: np.ndarray,
    w_true: List[float],
    w_init: List[float] = [29.0, 3.0],
    lr: float = 0.1,
    epochs: int = 10000,
) -> None:
    """Выполняет алгоритм подбора оптимальных параметров модели.

    Parameters:
        model: линейная модель f(x) = x @ w.T.
        xs: данные, которые подаются на вход модели.
        ys: данные, которые мы ожидает от модели получить.
        w_true: вектор (w_0, w_1) с реальными параметрами модели.
        w_init: вектор (w_0, w_1) с начальными параметрами модели.
        lr: (по умол. 0.1), шаг оптимизации.
        epochs: (по умол. 10000), кол-во эпох.

    Returns:
        None
    """
    w = np.array(w_init, dtype=np.float32)   # (2,)
    lr = float(lr)
    data_size = ys.size

    for epoch in range(1, epochs + 1):
        # Тренировка.
        # Прямой проход.
        preds = model(xs, w)

        # Подсчёт функции потерь MSE.
        errors = ys - preds
        train_loss = np.mean(errors**2)

        # Обратный проход.
        dw_2 = -2.0 * np.mean(xs[:, 1] * errors)
        dw_1 = -2.0 * np.mean(errors)
        dw = np.array([dw_1, dw_2], dtype=np.float32)

        # dw = -2.0 * (xs.T @ errors / data_size)

        print(f"Epoch {epoch:2d}:  train_loss={train_loss:.2f},  w={w.round(4)},  dw={dw.round(4)}")

        # Шаг оптимизации.
        print("           Корректировка обучаемых параметров ...")
        w = w - lr*dw

        # Валидация.
        # Прямой проход.
        preds = model(xs, w)

        # Подсчёт функции потерь MSE.
        errors = ys - preds
        val_loss = np.mean(errors**2)

        print(f"           val_loss={val_loss:.2f},  w={w.round(4)}", end="\n\n")

    print("\nReal parameters:")
    print(f"w_true = {w_true}")
    print("\nPredicted parameters:")
    print(f"w = {w.round(4)}")



In [ ]:
# Устанавливаем seed для воспроизведения.
seed = 1961

# Подготовка данных.
w_true = [32.0, 1.8]    # [w0, w1]
xs, ys = creating_data(
    w_true,
    x_min=-40.0,
    x_max=40.0,
    data_size=2000,
    noise_std=1,
    seed=seed
)

# Создание модели.
model = linear_model

# Цикл тренировки.
train(
    model,
    xs, ys,
    w_true,
    w_init=[29.0, 1.3], # [w0, w1]
    lr=0.001,
    epochs=10,
)



### SGD - Stochastic Gradient Descent

In [ ]:
def creating_data(
    w: list[float],
    x_min: float = -40.0,
    x_max: float = 40.0,
    data_size: int = 100,
    noise_std: float = 0.0,
    is_outlier: bool = False,
    seed: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """Вернёт данные разметки с нормальным распределением ошибки измерений.

    Parameters:
        w: вектор (w_0, w_1) с реальными параметрами модели.
        x_min: (по умол. -40.0), минимальное значение входных данных.
        x_max: (по умол. 40.0), максимальное значение входных данных.
        data_size: (по умол. 100), кол-во наблюдений в наборе данных.
        noise_std: (по умол. 0.0), стандартное отклонение ошибки.
        is_outlier: (по умол. False), добавлять выбросы в данные или нет.
        seed: (по умол. None), значение для инициализации генератора 
    случайных чисел.

    Returns:
        Tuple[np.ndarray, np.ndarray]
    """
    rng = np.random.default_rng(seed)

    x = np.linspace(x_min, x_max, data_size, dtype=np.float32)  # (data_size,)
    xs = np.stack((np.ones_like(x), x), axis=1)   # (data_size, 2)
    ys = xs @ np.array(w, dtype=np.float32)    # (data_size,)

    # Добавляем ошибку в данные.
    if noise_std > 0:
        ys += rng.normal(scale=noise_std, size=ys.shape).astype(np.float32)

    # Добавляем выбросы в данные.
    if is_outlier:
        idx = rng.choice(data_size, int(data_size*0.15), replace=False)
        ys[idx] += 50
        
    return xs, ys


def linear_model(x: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Линейная модель f(x) = x @ w.T."""
    return x @ w.T


def train_mse(
    model,
    xs: np.ndarray,
    ys: np.ndarray,
    w_true: List[float],
    w_init: List[float] = [29.0, 3.0],
    batch: int = 32,
    lr: float = 0.1,
    epochs: int = 10000,
    shuffle: bool = False,
    seed: Optional[int] = None
) -> None:
    """Выполняет алгоритм подбора оптимальных параметров модели.
    
    Parameters:
        model: линейная модель f(x) = x @ w.T.
        xs: данные, которые подаются на вход модели.
        ys: данные, которые мы ожидает от модели получить.
        w_true: вектор с реальными параметрами модели.
        w_init: вектор с начальными параметрами модели.
        batch: (по умол. 32), размер батча.
        lr: (по умол. 0.1), шаг оптимизации.
        epochs: (по умол. 10000), кол-во эпох.
        shuffle: (по умол. False), перемешивать данные или нет.
        seed: (по умол. None), значение для инициализации генератора 
    случайных чисел.

    Returns:
        None
    """
    w = np.array(w_init, dtype=np.float32)   # (2,)
    batch = int(batch)
    lr = float(lr)
    data_size, features = xs.shape

    # Количество батчей в наборе данных.
    num_batches = data_size // batch
    if num_batches == 0:
        raise ValueError(f"Размер батча не должен превышать {data_size}")

    print(f"Размер батча = {batch}")
    print(f"Кол-во целых батчей = {num_batches}", end="\n\n")

    total = num_batches * batch

    val_loss_history = []
    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch:2d}:")

        # Если нужно, то перемешиваем данные.
        if shuffle:
            rng = np.random.default_rng(seed+epoch)
            # Формируем индексы в рандомном порядке.
            idx = rng.permutation(data_size)
            # Перемешиваем данные.
            xs = xs[idx]
            ys = ys[idx]

        xs_batches = xs[:total].reshape(num_batches, batch, features)
        ys_batches = ys[:total].reshape(num_batches, batch)

        # Тренировка.
        for i, (x, y) in enumerate(zip(xs_batches, ys_batches)):
            # Прямой проход.
            preds = model(x, w)   # (batch,)

            # Подсчёт функции потерь MSE.
            errors = y - preds   # (batch,)
            train_loss = np.mean(errors**2)

            # Обратный проход.
            # dw_2 = -2.0 * np.mean(x[:, 1] * errors)
            # dw_1 = -2.0 * np.mean(errors)
            # dw = np.array([dw_1, dw_2])   # (2,)

            dw = -2.0 * (errors @ x / batch)   # (2,)

            print(f"    Batch {i+1:3d}:  train_loss={train_loss:.2f},  w={w.round(4)},  dw={dw.round(4)}")

            # Шаг оптимизации.
            w = w - lr*dw   # (2,)

        # Валидация.
        # Прямой проход.
        preds = model(xs, w)

        # Подсчёт функции потерь MSE.
        errors = ys - preds
        val_loss = np.mean(errors**2)

        # Сохраняю промежуточные результаты.
        val_loss_entry = {
            'epoch': epoch,
            'loss_val': val_loss,
            'w': w
        }
        val_loss_history.append(val_loss_entry)


        print(f"               val_loss={val_loss:.2f},  w={w.round(4)}", end="\n\n")

    print("\033[32mРезультаты обучения:\033[0m")
    for data in val_loss_history:
        print(f"Epoch {data['epoch']:3d}: loss_val = {data['loss_val']:.2f},  w = {data['w'].round(4)}")

    print("\nReal parameters:")
    print(f"w_true = {w_true}")
    print("\nPredicted parameters:")
    print(f"w = {w.round(4)}")



In [ ]:
# Устанавливаем seed для воспроизведения.
seed = 1961

# Подготовка данных.
w_true = [32.0, 1.8]    # [w0, w1]
xs, ys = creating_data(
    w_true,
    x_min=-40.0,
    x_max=40.0,
    data_size=2000,
    noise_std=1,
    is_outlier=True,
    seed=seed
)

# Создание модели.
model = linear_model

# Цикл тренировки.
train_mse(
    model,
    xs, ys,
    w_true,
    w_init=[29.0, 1.3], # [w0, w1]
    batch=16,
    lr=0.001,
    epochs=50,
    shuffle=True,
    seed=seed,
)



In [ ]:
def train_mae(
    model,
    xs: np.ndarray,
    ys: np.ndarray,
    w_true: List[float],
    w_init: List[float] = [29.0, 3.0],
    batch: int = 32,
    lr: float = 0.1,
    epochs: int = 10000,
    shuffle: bool = False,
    seed: Optional[int] = None
) -> None:
    """Выполняет алгоритм подбора оптимальных параметров модели.
    
    Parameters:
        model: линейная модель f(x) = x @ w.T.
        xs: данные, которые подаются на вход модели.
        ys: данные, которые мы ожидает от модели получить.
        w_true: вектор с реальными параметрами модели.
        w_init: вектор с начальными параметрами модели.
        batch: (по умол. 32), размер батча.
        lr: (по умол. 0.1), шаг оптимизации.
        epochs: (по умол. 10000), кол-во эпох.
        shuffle: (по умол. False), перемешивать данные или нет.
        seed: (по умол. None), значение для инициализации генератора 
    случайных чисел.

    Returns:
        None
    """
    w = np.array(w_init, dtype=np.float32)   # (2,)
    batch = int(batch)
    lr = float(lr)
    data_size, features = xs.shape

    # Количество батчей в наборе данных.
    num_batches = data_size // batch
    if num_batches == 0:
        raise ValueError(f"Размер батча не должен превышать {data_size}")

    print(f"Размер батча = {batch}")
    print(f"Кол-во целых батчей = {num_batches}", end="\n\n")

    total = num_batches * batch

    val_loss_history = []
    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch:2d}:")

        # Если нужно, то перемешиваем данные.
        if shuffle:
            rng = np.random.default_rng(seed+epoch)
            # Формируем индексы в рандомном порядке.
            idx = rng.permutation(data_size)
            # Перемешиваем данные.
            xs = xs[idx]
            ys = ys[idx]

        xs_batches = xs[:total].reshape(num_batches, batch, features)
        ys_batches = ys[:total].reshape(num_batches, batch)

        # Тренировка.
        for i, (x, y) in enumerate(zip(xs_batches, ys_batches)):
            # Прямой проход.
            preds = model(x, w)   # (batch,)

            # Подсчёт функции потерь MAE.
            errors = np.abs(y - preds)   # (batch,)
            sign = -1*np.sign(y - preds)
            train_loss = np.mean(errors)

            # Обратный проход.
            dw = np.mean(x*sign[:, None], axis=0)   # (2,)

            print(f"    Batch {i+1:3d}:  train_loss={train_loss:.2f},  w={w.round(4)},  dw={dw.round(4)}")

            # Шаг оптимизации.
            w = w - lr*dw   # (2,)

        # Валидация.
        # Прямой проход.
        preds = model(xs, w)

        # Подсчёт функции потерь MAE.
        errors = np.abs(ys - preds)
        val_loss = np.mean(errors)

        # Сохраняю промежуточные результаты.
        val_loss_entry = {
            'epoch': epoch,
            'loss_val': val_loss,
            'w': w
        }
        val_loss_history.append(val_loss_entry)


        print(f"               val_loss={val_loss:.2f},  w={w.round(4)}", end="\n\n")

    print("\033[32mРезультаты обучения:\033[0m")
    for data in val_loss_history:
        print(f"Epoch {data['epoch']:3d}: loss_val = {data['loss_val']:.2f},  w = {data['w'].round(4)}")

    print("\nReal parameters:")
    print(f"w_true = {w_true}")
    print("\nPredicted parameters:")
    print(f"w = {w.round(4)}")



In [ ]:
# Создание модели.
model = linear_model

# Цикл тренировки.
train_mae(
    model,
    xs, ys,
    w_true,
    w_init=[29.0, 1.3], # [w0, w1]
    batch=16,
    lr=0.001,
    epochs=50,
    shuffle=True,
    seed=seed,
)